# Decision Trees and K-Means with FHE

This notebook demonstrates how to use Decision Trees and K-Means clustering
with the Xcapit FHE-ML SDK for privacy-preserving machine learning.

## Topics Covered
1. Soft Decision Trees for classification
2. K-Means clustering with soft assignments
3. Training on plaintext, predicting on encrypted data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, silhouette_score

# SDK imports
import sys
sys.path.insert(0, '..')

from sdk.models import (
    DecisionTree,
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    TreeConfig,
    KMeans,
    MiniBatchKMeans,
    KMeansConfig,
    InitMethod,
)

## 1. Decision Tree Classification

Our FHE-compatible Decision Trees use "soft" splits based on sigmoid functions,
which allows gradient-based training and compatibility with homomorphic encryption.

In [ ]:
# Generate classification data
X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_classes=2,
    random_state=42
)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")

In [ ]:
# Configure the decision tree
config = TreeConfig(
    max_depth=4,           # Tree depth
    learning_rate=0.1,     # For gradient descent
    n_epochs=50,           # Training iterations
    temperature=1.0,       # Softness of splits (higher = softer)
)

# Create and train classifier
tree = DecisionTreeClassifier(config=config)
tree._fit_plaintext(X_train, y_train)

print(f"Training complete!")
print(f"Final loss: {tree.history.losses[-1]:.4f}")

In [ ]:
# Make predictions
y_pred = tree._predict_plaintext(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Get probability predictions
y_proba = tree._predict_proba_plaintext(X_test)
print(f"\nSample predictions (first 5):")
for i in range(5):
    print(f"  Sample {i}: True={y_test[i]}, Pred={y_pred[i]}, P(class=1)={y_proba[i, 1]:.3f}")

## 2. K-Means Clustering

Our K-Means implementation uses soft cluster assignments via softmax,
making it compatible with FHE operations.

In [ ]:
# Generate clustering data
X_cluster, y_true = make_blobs(
    n_samples=300,
    n_features=5,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

# Normalize
scaler = StandardScaler()
X_cluster = scaler.fit_transform(X_cluster)

print(f"Samples: {len(X_cluster)}")
print(f"True clusters: {len(np.unique(y_true))}")

In [ ]:
# Configure K-Means
kmeans_config = KMeansConfig(
    n_clusters=3,
    max_iter=100,
    init_method=InitMethod.KMEANS_PLUS_PLUS,  # Smart initialization
    tol=1e-4,
)

# Create and fit
kmeans = KMeans(config=kmeans_config)
kmeans._fit_plaintext(X_cluster)

print(f"Converged in {kmeans.n_iter} iterations")
print(f"Inertia: {kmeans.inertia:.4f}")

In [ ]:
# Get cluster assignments
labels = kmeans._predict_plaintext(X_cluster)

# Evaluate
silhouette = silhouette_score(X_cluster, labels)
print(f"Silhouette Score: {silhouette:.4f}")

# Show cluster sizes
unique, counts = np.unique(labels, return_counts=True)
print(f"\nCluster sizes:")
for cluster, count in zip(unique, counts):
    print(f"  Cluster {cluster}: {count} samples")

In [ ]:
# Get soft assignments (membership probabilities)
soft_assignments = kmeans._transform_plaintext(X_cluster)

print("Soft assignments (first 5 samples):")
for i in range(5):
    probs = soft_assignments[i]
    print(f"  Sample {i}: {probs.round(3)} -> Cluster {labels[i]}")

## 3. Mini-Batch K-Means

For larger datasets, use MiniBatchKMeans for faster convergence.

In [ ]:
# Larger dataset
X_large, _ = make_blobs(
    n_samples=2000,
    n_features=10,
    centers=5,
    random_state=42
)
X_large = StandardScaler().fit_transform(X_large)

# Mini-batch K-Means
mb_kmeans = MiniBatchKMeans(
    n_clusters=5,
    batch_size=100,
    max_iter=100,
)
mb_kmeans._fit_plaintext(X_large)

labels_large = mb_kmeans._predict_plaintext(X_large)
silhouette_large = silhouette_score(X_large, labels_large)

print(f"Mini-batch K-Means:")
print(f"  Samples: {len(X_large)}")
print(f"  Silhouette: {silhouette_large:.4f}")
print(f"  Inertia: {mb_kmeans.inertia:.4f}")

## 4. Using with Encrypted Data

In production, you would:
1. Train the model on plaintext data (secure environment)
2. Deploy model parameters
3. Make predictions on encrypted client data

```python
# Example workflow with encryption
from sdk.encryption import FHEContextManager, CKKSEncryptor
from sdk.utils import SecureDataLoader

# Setup encryption
manager = FHEContextManager()
manager.generate_context(poly_modulus_degree=8192)
encryptor = CKKSEncryptor(manager)

# Encrypt client data
loader = SecureDataLoader(encryptor)
encrypted_X = loader.encrypt_matrix(client_data)

# Predict on encrypted data
encrypted_predictions = tree.predict(encrypted_X)

# Client decrypts results
predictions = encryptor.decrypt_vector(encrypted_predictions)
```

## Summary

- **DecisionTreeClassifier**: Soft decision trees using sigmoid splits
- **KMeans**: Soft cluster assignments via softmax
- **MiniBatchKMeans**: Scalable clustering for large datasets

All models are trained on plaintext but can make predictions on encrypted data,
enabling privacy-preserving inference.